# acc

In [3]:
import json

# 文件路径
qa_pairs_path = '/root/repo/MindSearch/Datasets/qa_pairs.json'
mindsearch_results_path = '/root/repo/MindSearch/Datasets/mindsearch_results_5.json'

# 读取 JSON 文件
with open(qa_pairs_path, 'r') as f:
    qa_pairs = json.load(f)

with open(mindsearch_results_path, 'r') as f:
    mindsearch_results = json.load(f)

# 创建一个字典来存储问题和答案对
qa_dict = {item['question']: item['answer'] for item in qa_pairs}

# 比较答案
results = []
correct_count = 0

for result in mindsearch_results:
    query = result.get('query', '')
    response = result.get('responses', '')
    actual_answer = response
    
    # # 提取实际回答
    # if isinstance(response, dict):
    #     actual_answer = response.get('response', {}).get('content', '')
    # else:
    #     actual_answer = response
    
    # 获取预期答案
    expected_answer = qa_dict.get(query, '')

    # 判断答案是否一致
    if expected_answer and actual_answer:
        # is_correct = expected_answer.lower() in str(actual_answer).lower()
        is_correct = expected_answer in actual_answer
    else:
        is_correct = False
    
    if is_correct:
        correct_count += 1
    
    results.append({
        'query': query,
        'expected_answer': expected_answer,
        'actual_answer': actual_answer,
        'is_correct': is_correct
    })

# 计算准确度
total_questions = len(mindsearch_results)
accuracy = correct_count / total_questions if total_questions > 0 else 0


# 筛选回答错误的问题
incorrect_results = [result for result in results if not result['is_correct']]

# 输出回答错误的问题
print("\nIncorrect Results:")
for result in incorrect_results:
    print(f"Query: {result['query']}")
    print(f"Expected Answer: {result['expected_answer']}")
    print(f"Actual Answer: {result['actual_answer']}")
    print(f"Is Correct: {result['is_correct']}")
    print('-' * 50)

# # 输出结果
# for result in results:
#     print(f"Query: {result['query']}")
#     print(f"Expected Answer: {result['expected_answer']}")
#     print(f"Actual Answer: {result['actual_answer']}")
#     print(f"Is Correct: {result['is_correct']}")
#     print('-' * 50)

print(f"Total Questions: {total_questions}")
print(f"Correct Answers: {correct_count}")
print(f"Incorrect Answers: {total_questions - correct_count}")
print(f"Accuracy: {accuracy:.2%}")


Incorrect Results:
Query: When did the last king from Britain's House of Hanover die?
Expected Answer: 20 June 1837
Actual Answer: The last king from Britain's House of Hanover was George V. He reigned as the King of Hanover from November 18, 1851, to September 20, 1866 [[1]][[2]]. King George V died on January 20, 1936. His death occurred at Sandringham House in Norfolk, England [[4]][[5]][[6]]. He had been suffering from chronic lung issues since 1928 and his health had gradually declined over the previous few months [[5]]. The exact cause of his death was a mixture of cocaine and morphine administered by his physician, Lord Bertrand Dawson, who intended to grant him a painless death [[7]][[8]].

Thus, the last king from Britain's House of Hanover died on January 20, 1936.
Is Correct: False
--------------------------------------------------
Query: How many people died in the second most powerful earthquake ever recorded?
Expected Answer: 131
Actual Answer: The second most powerful e

## 检测空答案
前期没有更改 prompt 以及 graph 的话，会因为无法分解单跳问题而导致回答为空，修改 prompt 当为单挑问题时，直接搜索 root 节点的原始问题即可。
还有 LLM 在生成代码的时候，经常不创建(add_node)就连接边(add_edge)和查看节点(nodes)，导致报错。
还有一种情况，查看节点(nodes())的时候， LLM 会出现幻觉，生成近义词节点名词，和之前创建和添加边的名词不一致，导致 Keyerror ，解决办法是在 graph 里面把从 nodes() 代码检测节点列表改成从 add_node() 代码检测节点列表，甚至我觉得 nodes 函数可以删除了，反正必须执行这个函数，那不如 add_node() 的时候就让 websearcher 搜索创建节点的 content 提问。

In [1]:
import json

# 加载数据
file_path = "/root/repo/MindSearch/Datasets/mindsearch_results.json"
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# 统计答案为空的数量
empty_responses_count = sum(1 for item in data if not item["responses"])
total_questions = len(data)

# 计算比例
empty_responses_ratio = empty_responses_count / total_questions if total_questions > 0 else 0

# 输出结果
print(f"总问题数: {total_questions}")
print(f"答案为空的问题数: {empty_responses_count}")
print(f"答案为空的问题比例: {empty_responses_ratio:.2%}")

总问题数: 125
答案为空的问题数: 9
答案为空的问题比例: 7.20%


# 要改进这个基于 LLM 的 AI Agent 搜索引擎，可以从以下几个方面入手：

---

### 1. **改进问题拆解与图构建逻辑**
当前系统通过 LLM 生成代码来构建搜索图，但可能存在以下问题：
- **问题拆解不够细化**：复杂问题可能没有被充分拆解成独立的子问题。
- **节点与边的生成顺序问题**：可能会出现边生成时节点尚未生成的情况。

**改进建议**：
- **增强 Prompt 设计**：在 Prompt 中明确要求 LLM 生成的代码必须先添加节点 (`add_node`)，再添加边 (`add_edge`)。
- **自动化节点补全**：在 `add_edge` 方法中，自动补全缺失的节点，避免因节点缺失导致的错误。
- **优化问题拆解策略**：引入基于规则或额外模型的辅助模块，帮助 LLM 更好地拆解复杂问题。例如：
  - 使用专门的 NLP 模型（如 T5 或 GPT）来分析问题并生成子问题。
  - 在 Prompt 中加入更多的 Few-shot 示例，展示如何拆解复杂问题。

---

### 2. **提升搜索引擎的能力与集成**
当前系统依赖于搜索引擎 API（如 Bing 或 TencentSearch）获取信息，但可能存在以下问题：
- **搜索结果质量不稳定**：不同搜索引擎的结果可能不一致。
- **缺乏多模态支持**：无法处理图片、视频等非文本信息。

**改进建议**：
- **多搜索引擎集成**：同时调用多个搜索引擎（如 Google、Bing、DuckDuckGo），并通过结果聚合算法（如投票机制或置信度加权）提升结果质量。
- **引入多模态搜索**：支持图片、视频等非文本信息的搜索。例如：
  - 使用 Google Vision API 或其他图像搜索引擎。
  - 集成 YouTube 或其他视频平台的 API。
- **上下文增强搜索**：在搜索时结合上下文信息（如历史查询、当前节点内容）生成更精准的查询。

---

### 3. **改进搜索结果的处理与引用**
当前系统通过搜索引擎返回的结果生成节点内容，但可能存在以下问题：
- **结果可信度不足**：引用的来源可能不够权威。
- **结果冗余或不相关**：搜索结果可能包含大量无关信息。

**改进建议**：
- **结果过滤与排序**：引入基于 NLP 的结果过滤模块，筛选出与问题最相关的内容。例如：
  - 使用 BERT 或 GPT 对搜索结果进行相关性评分。
  - 过滤掉低质量或重复的结果。
- **结果可信度评估**：为每个搜索结果打分，优先使用权威来源（如 Wikipedia、政府网站等）。
- **引用优化**：改进引用格式，确保每个答案都清晰标注来源，并支持多来源引用。

---

### 4. **提升 LLM 的生成与执行效率**
当前系统依赖 LLM 生成代码并通过 `exec` 执行，可能存在以下问题：
- **生成效率低**：LLM 生成代码可能较慢，尤其是对于复杂问题。
- **执行效率低**：生成的代码可能不够优化，导致执行效率低下。

**改进建议**：
- **代码生成优化**：在 Prompt 中加入更多优化代码的示例，指导 LLM 生成更高效的代码。
- **缓存机制**：对常见问题的结果进行缓存，避免重复生成和执行代码。
- **并行化执行**：对于独立的子问题，支持并行生成和执行代码。

---

### 5. **增强用户交互与可解释性**
当前系统的交互方式可能较为单一，用户难以理解搜索图的构建过程。

**改进建议**：
- **可视化搜索图**：改进 `create_network_graph` 和 `draw_graph` 方法，生成更直观的图形界面，展示问题拆解与搜索过程。
- **交互式查询**：允许用户手动调整搜索图，例如添加或删除节点、修改边的连接。
- **结果解释**：为每个答案提供详细的解释，包括搜索过程、引用来源和推理逻辑。

---

### 6. **提升系统的鲁棒性与容错性**
当前系统可能在以下情况下出现问题：
- **搜索引擎 API 调用失败**。
- **LLM 生成的代码有语法错误或逻辑错误**。

**改进建议**：
- **错误处理机制**：在搜索引擎调用失败或代码执行出错时，提供友好的错误提示，并尝试自动修复。例如：
  - 在 `ExecutionAction.run` 方法中捕获异常并记录详细日志。
  - 在搜索引擎调用失败时，自动切换到备用搜索引擎。
- **代码验证与修复**：在执行 LLM 生成的代码之前，使用静态分析工具（如 `ast` 模块）检查代码的语法和逻辑。

---

### 7. **支持多语言与多文化背景**
当前系统可能主要支持中文或英文，其他语言的支持可能较弱。

**改进建议**：
- **多语言支持**：扩展 Prompt 和搜索引擎 API，支持更多语言（如法语、西班牙语等）。
- **文化背景适配**：根据用户的文化背景调整搜索策略和结果呈现方式。例如：
  - 针对不同地区的用户，优先使用本地化的搜索引擎。
  - 在结果中加入与用户文化相关的背景信息。

---

### 8. **引入领域知识与专用模型**
对于特定领域的问题（如医学、法律、金融），通用搜索引擎和 LLM 的表现可能不足。

**改进建议**：
- **领域知识库**：集成领域知识库（如 PubMed、LexisNexis），提升专业问题的回答质量。
- **专用 LLM**：针对特定领域训练专用 LLM（如 BioGPT、LegalBERT）。

---

### 9. **性能与扩展性优化**
随着用户量的增加，系统可能面临性能瓶颈。

**改进建议**：
- **分布式架构**：将 LLM 推理、搜索引擎调用和图构建分布到不同的服务器上，提升系统的并发能力。
- **异步处理**：充分利用异步框架（如 `asyncio`），提升任务处理效率。
- **负载均衡**：在多台服务器之间分配请求，避免单点瓶颈。

---

通过以上改进，可以显著提升系统的搜索能力、用户体验和整体性能，使其更适合实际应用场景。

# 在涉及排名、时间日期、数字数据等动态信息时，得不到想要的答案可能是以下几个原因导致的：

---

### 1. **搜索引擎信息来源问题**
- **问题**：
  - 搜索引擎返回的信息可能不够权威或准确，尤其是当搜索引擎的结果中包含过时或不相关的信息时。
  - 搜索引擎可能未能返回最新的数据，特别是对于动态变化的信息（如排名、时间、数字等）。
- **验证方法**：
  - 检查搜索引擎返回的原始结果，确认是否包含相关信息。
  - 对比多个搜索引擎（如 Google、Bing、DuckDuckGo）的结果，验证是否存在一致性问题。
- **改进建议**：
  - **多搜索引擎集成**：同时调用多个搜索引擎，并通过结果聚合算法（如投票机制或置信度加权）提升结果质量。
  - **优先权威来源**：在搜索结果中优先选择权威来源（如 Wikipedia、政府网站、学术数据库等）。
  - **实时性增强**：对于动态信息，优先选择实时更新的搜索引擎或 API。

---

### 2. **数据集过时问题**
- **问题**：
  - 如果搜索引擎的索引数据较旧，或者 LLM 的预训练数据未包含最新信息，可能导致答案不准确。
- **验证方法**：
  - 对比搜索引擎返回的结果与最新的权威数据，确认是否存在时间差异。
  - 检查 LLM 的生成内容是否基于过时的知识。
- **改进建议**：
  - **更新搜索引擎索引**：确保搜索引擎使用最新的索引数据。
  - **增强 LLM 的后处理能力**：通过后处理逻辑（如时间过滤）剔除过时信息。
  - **动态知识注入**：通过插件或外部知识库（如实时 API）为 LLM 提供最新信息。

---

### 3. **图生成结构问题**
- **问题**：
  - 图的生成逻辑可能未能正确拆解问题，导致搜索节点的内容不够具体或相关性不足。
  - 图的结构可能未能充分利用搜索结果，导致信息整合不完整。
- **验证方法**：
  - 检查生成的图结构，确认是否正确拆解了问题。
  - 检查每个节点的内容是否具体且与问题相关。
  - 检查边的连接是否合理，是否存在冗余或缺失的节点。
- **改进建议**：
  - **优化问题拆解逻辑**：在 Prompt 中明确要求 LLM 将问题拆解为具体的子问题，避免复合问题。
  - **增强图的动态调整能力**：在生成图时，根据搜索结果动态调整节点和边的内容。
  - **改进节点生成顺序**：确保 `add_node` 的调用先于 `add_edge`，并在调用 `add_edge` 时检查节点是否存在。

---

### 4. **LLM 的生成能力问题**
- **问题**：
  - LLM 可能未能正确理解问题，导致生成的代码或搜索内容不符合预期。
  - LLM 的生成内容可能缺乏对动态信息的处理能力。
- **验证方法**：
  - 检查 LLM 生成的代码，确认是否正确调用了 `add_node` 和 `add_edge`。
  - 检查 LLM 生成的搜索内容是否具体且相关。
- **改进建议**：
  - **增强 Prompt 设计**：在 Prompt 中加入更多 Few-shot 示例，指导 LLM 生成更高质量的代码。
  - **动态信息处理模块**：为 LLM 提供专门处理动态信息的模块（如时间解析、排名计算等）。
  - **后处理逻辑优化**：在生成结果后，通过后处理逻辑（如正则表达式、过滤器）提取和验证动态信息。

---

### 5. **搜索结果的整合与引用问题**
- **问题**：
  - 搜索结果可能未被正确整合，导致答案不完整或不准确。
  - 搜索结果的引用可能不够清晰，导致用户难以验证答案的可信度。
- **验证方法**：
  - 检查搜索结果的整合逻辑，确认是否正确提取了相关信息。
  - 检查答案中的引用是否清晰且与搜索结果一致。
- **改进建议**：
  - **结果过滤与排序**：引入基于 NLP 的结果过滤模块，筛选出与问题最相关的内容。
  - **引用优化**：改进引用格式，确保每个答案都清晰标注来源，并支持多来源引用。
  - **结果可信度评估**：为每个搜索结果打分，优先使用高可信度的来源。

---

### 综合改进方案
1. **多搜索引擎集成**：
   - 在 `process_query` 函数中，调用多个搜索引擎，并通过结果聚合算法提升结果质量。
   - 示例代码：
     ```python
     def process_query(query, urls=["http://search1.com", "http://search2.com"]):
         results = []
         for url in urls:
             response = requests.post(url, headers=headers, data=json.dumps(data), timeout=20)
             results.append(response.json())
         # 聚合结果
         aggregated_results = aggregate_results(results)
         return aggregated_results
     ```

2. **动态知识注入**：
   - 在 `WebSearchGraph.add_node` 方法中，调用实时 API 获取最新信息。
   - 示例代码：
     ```python
     def add_node(self, node_name: str, node_content: str):
         if "date" in node_content.lower():
             # 调用实时 API 获取最新日期信息
             response = requests.get("http://worldclockapi.com/api/json/utc/now")
             node_content += f" (Updated: {response.json()['currentDateTime']})"
         self.nodes[node_name] = dict(content=node_content, type="searcher")
     ```

3. **优化 Prompt 设计**：
   - 在 mindsearch_prompt.py 中，加入更多动态信息处理的示例。
   - 示例：
     ```python
     GRAPH_PROMPT_CN = """
     ## 示例
     问题：2024年最受欢迎的编程语言是什么？
     1. 添加节点：搜索2024年编程语言排名。
     2. 添加节点：搜索编程语言的受欢迎程度。
     3. 整合结果，生成最终答案。
     """
     ```

4. **结果可信度评估**：
   - 在 `streaming` 函数中，为每个搜索结果打分，并优先选择高分结果。
   - 示例代码：
     ```python
     def streaming(raw_response):
         for chunk in raw_response.iter_lines():
             response = json.loads(chunk.decode("utf-8"))
             response["score"] = evaluate_response(response)
             yield response
     ```

---

通过以上验证和改进，可以有效提升系统在处理动态信息（如排名、时间、数字数据）时的表现，确保答案的准确性和可信度。

# 要实现图结构的复用、更新和纠错功能，可以从以下几个方面进行改进：

---

### 1. **引入图的持久化与加载功能**
为了在后续问题中复用之前生成的图，需要将图的结构（节点和边）持久化到文件或数据库中，并在新问题到来时加载已有的图。

#### 修改 `WebSearchGraph` 类
在 `WebSearchGraph` 类中添加保存和加载图的方法：



In [ ]:
import json

class WebSearchGraph:
    # ... existing code ...

    def save_graph(self, file_path: str):
        """保存图结构到文件"""
        graph_data = {
            "nodes": self.nodes,
            "adjacency_list": self.adjacency_list,
        }
        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(graph_data, f, ensure_ascii=False, indent=4)

    def load_graph(self, file_path: str):
        """从文件加载图结构"""
        with open(file_path, "r", encoding="utf-8") as f:
            graph_data = json.load(f)
        self.nodes = graph_data.get("nodes", {})
        self.adjacency_list = graph_data.get("adjacency_list", defaultdict(list))



#### 在 `process_query` 中加载图
在处理新问题时，检查是否存在之前保存的图，如果存在则加载：



In [ ]:
def process_query(query, url="http://localhost:8002/solve", graph_file="graph.json"):
    headers = {"Content-Type": "application/json"}
    data = {"inputs": query}
    raw_response = requests.post(url, headers=headers, data=json.dumps(data), timeout=20, stream=True)

    graph = WebSearchGraph()
    # 加载已有图
    try:
        graph.load_graph(graph_file)
        print("Loaded existing graph.")
    except FileNotFoundError:
        print("No existing graph found. Starting a new graph.")
        graph.add_root_node(node_content=query)

    # ... existing logic to process query ...

    # 保存图
    graph.save_graph(graph_file)
    return responses, graph_html, graph.nodes, graph.adjacency_list



---

### 2. **支持图的更新与纠错**
在用户提出新问题时，可以动态更新图的节点和边，或者对已有节点进行纠错。

#### 添加更新节点和边的方法
在 `WebSearchGraph` 类中添加更新节点和边的方法：



In [ ]:
class WebSearchGraph:
    # ... existing code ...

    def update_node(self, node_name: str, new_content: str):
        """更新节点内容"""
        if node_name in self.nodes:
            self.nodes[node_name]["content"] = new_content
        else:
            raise KeyError(f"Node '{node_name}' does not exist.")

    def remove_node(self, node_name: str):
        """移除节点及其相关的边"""
        if node_name in self.nodes:
            del self.nodes[node_name]
            self.adjacency_list = {
                k: [n for n in v if n["name"] != node_name]
                for k, v in self.adjacency_list.items()
            }
        else:
            raise KeyError(f"Node '{node_name}' does not exist.")

    def remove_edge(self, start_node: str, end_node: str):
        """移除边"""
        if start_node in self.adjacency_list:
            self.adjacency_list[start_node] = [
                n for n in self.adjacency_list[start_node] if n["name"] != end_node
            ]
        else:
            raise KeyError(f"Start node '{start_node}' does not exist.")



#### 在 `process_query` 中支持更新逻辑
允许用户通过特定的输入格式（如 `"update:node_name:new_content"`）来更新图：



In [ ]:
def process_query(query, url="http://localhost:8002/solve", graph_file="graph.json"):
    headers = {"Content-Type": "application/json"}
    data = {"inputs": query}
    raw_response = requests.post(url, headers=headers, data=json.dumps(data), timeout=20, stream=True)

    graph = WebSearchGraph()
    try:
        graph.load_graph(graph_file)
    except FileNotFoundError:
        graph.add_root_node(node_content=query)

    # 检查是否是更新操作
    if query.startswith("update:"):
        _, node_name, new_content = query.split(":", 2)
        graph.update_node(node_name, new_content)
        graph.save_graph(graph_file)
        return ["Node updated successfully"], None, graph.nodes, graph.adjacency_list

    # ... existing logic to process query ...

    graph.save_graph(graph_file)
    return responses, graph_html, graph.nodes, graph.adjacency_list



---

### 3. **支持上下文记忆**
通过在 `WebSearchGraph` 中维护上下文信息，可以让后续问题基于已有的图结构继续提问。

#### 在 `WebSearchGraph` 中添加上下文方法
添加一个方法，用于获取当前图的上下文信息：



In [ ]:
class WebSearchGraph:
    # ... existing code ...

    def get_context(self):
        """获取当前图的上下文信息"""
        context = []
        for node_name, node_data in self.nodes.items():
            context.append(f"{node_name}: {node_data['content']}")
        return "\n".join(context)



#### 在 `process_query` 中使用上下文
在处理新问题时，将上下文信息传递给搜索引擎或 LLM：



In [ ]:
def process_query(query, url="http://localhost:8002/solve", graph_file="graph.json"):
    headers = {"Content-Type": "application/json"}
    graph = WebSearchGraph()
    try:
        graph.load_graph(graph_file)
    except FileNotFoundError:
        graph.add_root_node(node_content=query)

    # 获取上下文信息
    context = graph.get_context()
    data = {"inputs": f"{context}\n{query}"}
    raw_response = requests.post(url, headers=headers, data=json.dumps(data), timeout=20, stream=True)

    # ... existing logic to process query ...

    graph.save_graph(graph_file)
    return responses, graph_html, graph.nodes, graph.adjacency_list



---

### 4. **支持用户交互式纠错**
通过前端或命令行接口，允许用户手动修改图结构。

#### 示例交互逻辑


In [ ]:
def interactive_mode():
    graph = WebSearchGraph()
    try:
        graph.load_graph("graph.json")
    except FileNotFoundError:
        print("No existing graph found. Starting a new graph.")
        graph.add_root_node(node_content="Initial question")

    while True:
        command = input("Enter command (add/update/remove/show/exit): ").strip()
        if command == "exit":
            break
        elif command == "show":
            print(json.dumps({"nodes": graph.nodes, "adjacency_list": graph.adjacency_list}, indent=4))
        elif command.startswith("add:"):
            _, node_name, node_content = command.split(":", 2)
            graph.add_node(node_name=node_name, node_content=node_content)
        elif command.startswith("update:"):
            _, node_name, new_content = command.split(":", 2)
            graph.update_node(node_name, new_content)
        elif command.startswith("remove:"):
            _, node_name = command.split(":", 1)
            graph.remove_node(node_name)
        graph.save_graph("graph.json")



---

### 5. **总结**
通过以上改进，可以实现以下功能：
1. **图结构复用**：通过持久化和加载图结构，支持后续问题基于已有图继续提问。
2. **图节点更新与纠错**：支持动态更新节点内容、移除节点或边。
3. **上下文记忆**：通过维护上下文信息，让后续问题能够利用已有的图结构。
4. **交互式操作**：允许用户手动修改图结构，提升灵活性。

这些改进将显著提升系统的智能性和用户体验。

# 项目解析
这个项目 `MindSearch` 是一个基于 **网络搜索** 和 **自然语言处理** 的智能问答系统，结合了 **流式处理**、**图结构** 和 **多模型支持** 来组织和展示搜索结果。以下是对项目的详细解析，包括其架构、功能模块、数据流动和关键逻辑。

---

## **项目架构**

### 1. **核心功能**
- **网络搜索**：通过调用外部搜索工具（如 Bing、腾讯搜索等）获取相关信息。
- **自然语言生成**：基于搜索结果生成自然语言回答。
- **引用管理**：在回答中标注引用来源，并支持超链接跳转。
- **图结构展示**：将搜索结果和回答组织成图结构，便于可视化。
- **流式处理**：支持逐步生成回答，提高响应速度。

---

### 2. **主要模块**
#### **1. 后端核心模块**
- **agent**：
  - **`mindsearch_agent.py`**：
    - 定义了核心的 `MindSearchAgent` 和 `AsyncMindSearchAgent` 类，继承自 `StreamingAgentForInternLM`，实现了流式处理和引用生成。
    - 包含 `_update_ref` 和 `_generate_references_from_graph` 函数，用于处理回答中的引用和生成引用到 URL 的映射。
  - **`graph.py`**：
    - 定义了 `WebSearchGraph` 类，用于管理搜索结果的图结构。
    - 包含 `ExecutionAction` 类，用于执行搜索操作。
  - **`streaming.py`**：
    - 定义了 `StreamingAgentMixin` 和 `AsyncStreamingAgentMixin`，为代理提供流式处理能力。
    - 支持异步和同步的流式数据处理。
  - **`models.py`**：
    - 定义了模型配置，包括 InternLM、GPT-4 和其他模型的支持。
  - **`mindsearch_prompt.py`**：
    - 定义了搜索工具的提示模板（Prompt），包括中文和英文版本。
    - 指定了回答格式和引用标注规则。

- **`Datasets/backendpatch.py`**：
  - 实现了后端的查询处理逻辑，包括调用搜索工具、解析结果和生成回答。
  - 包含 `process_query` 函数，用于处理用户查询并返回回答和图结构。

#### **2. 前端模块**
- **`frontend/mindsearch_streamlit.py`**：
  - 使用 Streamlit 实现前端界面，支持用户输入查询、显示回答和可视化图结构。
  - 包含 `streaming` 函数，用于处理后端返回的流式数据。

#### **3. API 服务**
- **`mindsearch/app.py`**：
  - 使用 FastAPI 实现后端服务，提供 `/solve` 接口。
  - 支持同步和异步模式，调用 `MindSearchAgent` 处理用户请求。

---

## **功能解析**

### 1. **查询处理**
用户输入查询后，系统会调用 `process_query` 函数：
- **调用搜索工具**：通过 HTTP 请求调用后端搜索服务。
- **解析搜索结果**：将搜索结果组织成图结构，并提取相关内容。
- **生成回答**：基于搜索结果生成自然语言回答，并标注引用。

---

### 2. **引用管理**
- **引用格式**：回答中的引用以 `[[1]]` 或 `[[1]][[2]]` 的形式表示。
- **引用生成**：
  - `_update_ref`：更新引用编号并将其转换为超链接格式。
  - `_generate_references_from_graph`：从图结构中提取引用并生成引用到 URL 的映射。

---

### 3. **流式处理**
- **流式数据**：通过 `streaming` 函数逐步处理后端返回的数据块。
- **终止条件**：当 `stream_state` 为 `AgentStatusCode.END` 时，结束流式处理。

---

### 4. **图结构展示**
- **节点和边**：将搜索结果和回答组织成图结构，节点表示内容，边表示关系。
- **可视化**：使用 PyVis 库生成交互式图结构，并在前端展示。

---

## **数据流动**

### 1. **用户输入查询**
用户通过前端输入查询，调用 `process_query` 函数。

### 2. **后端处理查询**
- **调用搜索工具**：
  - 使用 `WebSearchGraph` 类管理搜索图。
  - 调用 `add_node` 添加搜索节点，调用 `add_edge` 添加边。
- **生成回答**：
  - 使用 `MindSearchAgent` 生成自然语言回答。
  - 调用 `_update_ref` 和 `_generate_references_from_graph` 处理引用。

### 3. **前端展示结果**
- **回答展示**：使用 Streamlit 显示回答。
- **图结构展示**：使用 PyVis 可视化图结构。

---

## **关键逻辑**

### 1. **引用生成逻辑**
#### `_update_ref` 函数：
- 使用正则表达式匹配 `[[数字]]` 格式的引用。
- 根据偏移量更新引用编号，并将其转换为超链接格式。

#### `_generate_references_from_graph` 函数：
- 遍历图结构中的节点，提取引用编号和 URL。
- 调用 `_update_ref` 更新引用并生成超链接。

---

### 2. **流式处理逻辑**
#### `streaming` 函数：
- 逐行解析后端返回的数据块。
- 提取当前节点、回答内容和图结构的邻接表。
- 根据 `stream_state` 判断是否继续处理。

---

### 3. **图结构生成逻辑**
#### `create_network_graph` 函数：
- 遍历节点和边，使用 PyVis 库生成图结构。
- 设置节点的颜色、大小和标签。

#### `draw_graph` 函数：
- 将生成的图结构保存为 HTML 文件，供前端展示。

---

## **改进建议**

1. **引用生成优化**：
   - 确保引用格式统一，避免多余字段（如 `summ` 和 `title`）出现在超链接中。

2. **终止条件检查**：
   - 在 `streaming` 函数中严格检查 `stream_state`，避免无限循环。

3. **错误处理**：
   - 增加对异常情况的处理，例如搜索结果为空或引用编号缺失。

4. **性能优化**：
   - 使用异步处理提高流式数据的处理效率。

---

## **总结**

`MindSearch` 是一个结合网络搜索和自然语言处理的智能问答系统，具有以下特点：
- **引用标注**：回答中标注引用来源，增强可信度。
- **流式处理**：逐步生成回答，提高响应速度。
- **图结构展示**：以交互式图结构展示搜索结果，便于理解。

通过进一步优化引用生成逻辑和流式处理，可以提升系统的稳定性和用户体验。